# 06 — GPT v1 on the Wizard of Oz

We replace the bigram's single table with a decoder-only transformer.

## Device (CUDA / Apple Silicon / CPU)
On a MacBook with Apple Silicon PyTorch uses the `mps` backend; the only code change is
the device string.

In [1]:
import pickle
import time
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(device)

cpu


## Overview of hyperparameters

| name | meaning |
|---|---|
| `batch_size` | sequences per step (parallelism, gradient noise) |
| `block_size` | context length — max tokens the model attends over |
| `max_iters` / `eval_iters` | training steps / batches averaged per evaluation |
| `learning_rate` | AdamW step size |
| `n_embd` | width of each token vector |
| `n_head` | attention heads per block (`head_size = n_embd // n_head`) |
| `n_layer` | number of transformer blocks |
| `dropout` | fraction of activations zeroed in training |

The values below are sized to train in a few minutes on a laptop CPU. On a GPU try
`batch_size=64, block_size=256, n_embd=384, n_head=8, n_layer=8`.

In [2]:
batch_size = 32
block_size = 64
max_iters = 3000
eval_interval = 500
learning_rate = 3e-4
eval_iters = 50
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.2
torch.manual_seed(1337)

In [3]:
with open('../data/wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
vocab_size = len(chars)
string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.8 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## Dot-product attention: one `Head`
`key`, `query`, `value` projections; scores scaled by `1/√head_size`; causal mask
registered as a **buffer** (moves with the model, but is not a parameter); dropout on
the attention weights.

In [4]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input (B, T, C) -> output (B, T, head_size)
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5          # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

## Multi-head attention
Heads live in an `nn.ModuleList` because they run **in parallel** on the same input
(not one after the other like `nn.Sequential`); results are concatenated along the
channel dim and projected back to `n_embd`.

In [5]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # (B, T, num_heads*head_size)
        return self.dropout(self.proj(out))

## FeedForward network
Per-token MLP: expand ×4, ReLU, project back. Attention *communicates*, the MLP *computes*.

In [6]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

## Transformer blocks
Pre-norm residual block: `x + sa(ln1(x))` then `x + ffwd(ln2(x))`.

In [7]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## GPTLanguageModel: initialization, positional encoding, forward pass

* token + **learned positional** embeddings are summed,
* `n_layer` Blocks in an `nn.Sequential`,
* final LayerNorm and `lm_head` producing `vocab_size` logits,
* `_init_weights`: normal(0, **std=0.02**) for Linear/Embedding weights, zero biases.

`generate` **crops** the context to the last `block_size` tokens — the positional table
has only `block_size` rows, so longer inputs would index out of range.

In [8]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape
        tok_emb = self.token_embedding_table(index)                                # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=index.device))  # (T, C)
        x = tok_emb + pos_emb                                                      # (B, T, C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                                   # (B, T, V)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]          # crop to the context window
            logits, _ = self.forward(index_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

model = GPTLanguageModel(vocab_size).to(device)
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters')

0.82M parameters


## Begin training

In [9]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
start = time.time()
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}  ({time.time() - start:.0f}s)")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step: 0, train loss: 4.435, val loss: 4.433  (2s)


step: 500, train loss: 2.045, val loss: 2.136  (58s)


step: 1000, train loss: 1.665, val loss: 1.803  (114s)


step: 1500, train loss: 1.527, val loss: 1.733  (171s)


step: 2000, train loss: 1.452, val loss: 1.646  (229s)


step: 2500, train loss: 1.392, val loss: 1.592  (287s)


step: 2999, train loss: 1.331, val loss: 1.556  (343s)


In [10]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=600)[0].tolist()))


It below crious good carly mind no perfor gleeps."

"I've to you othese some of this lighousily us," cymed a dear.

"It would
o quee that down his they pertded."You leaved into the gass for yyour
wardie untill them reat there beard.


Jum's fremust Unvite."

She gent kist, from place had afraine table to greins up the ene of holdam fright ven to delt
in that, or the Prince piglets of the
on and the adven buggy. Wizard to his was etreed in me of oead. The
cronger Wazard's qXoy: "Tathorse, than the Wizard ustaned Dorothy, ruled apped the tup
inny, and I as the wood, rully roone ouse tall me litt


Loss drops well below the bigram's ≈2.47 and the samples contain real words, names from
the book and dialogue structure. Larger `n_embd`/`n_layer`/`block_size` and more steps
(on a GPU) keep improving it.

## Saving the model (pickling)
`pickle` serialises the whole Python object (architecture + weights). The class
definitions must be importable when loading. `torch.save(model.state_dict(), path)`
(weights only) is the more portable alternative.

In [11]:
with open('model-01.pkl', 'wb') as f:
    pickle.dump(model, f)
print('model saved')

model saved
